# 04 — NER Fine-Tuning on DNRTI

**Objective:** train a BERT NER model on the DNRTI dataset we prepared in notebook 01, and save the result to `models/ner_dnrti/` for notebook 05 (evaluation) and notebook 10 (the unified pipeline) to consume.

## What's different from CTI 101 notebook 05?

The *code shape* is almost identical — same tokenizer, same `BertForTokenClassification` head, same Trainer loop. That's the point: a clean NER recipe transfers from a toy dataset to a real one with minor changes.

What actually changes:

| | CTI 101 | CTI 201 (this notebook) |
|---|---|---|
| Data source | tiny hand-crafted examples | DNRTI — ~300 real CTI reports, BIO-tagged |
| Tag classes | ~7 | ~27 (13 entity types × B/I + O) |
| Training epochs | 3 (converged fast on toy data) | 5–8 (real data needs longer) |
| Expected F1 | high (easy task) | moderate — real class imbalance bites |

## What this notebook does

1. Load the DNRTI `DatasetDict` from `processed/dnrti/`
2. Tokenize + align BIO labels to BERT's subwords (same `word_ids()` trick as CTI 101 notebook 04)
3. Fine-tune `bert-base-cased` with `BertForTokenClassification`
4. Report training-time token-level metrics (span-level evaluation lives in notebook 05)
5. Save the model + tokenizer + tag vocabulary to `models/ner_dnrti/`

## Prerequisite

Notebook 01 must have been run end-to-end. This notebook does not re-parse DNRTI.

## Step 1 — Load the prepared dataset

In [1]:
from pathlib import Path
from datasets import load_from_disk

DATA_PATH = Path('processed/dnrti')
assert DATA_PATH.exists(), f'Run notebook 01 first — {DATA_PATH} does not exist.'

ds = load_from_disk(str(DATA_PATH))
print(ds)

# The ClassLabel feature on ner_tags stores the tag names we built in nb 01
tag_names = ds['train'].features['ner_tags'].feature.names
num_labels = len(tag_names)
id2label = {i: t for i, t in enumerate(tag_names)}
label2id = {t: i for i, t in id2label.items()}

print(f'\nTotal tag classes: {num_labels}')
print(f'Examples: {tag_names[:8]} ...')

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 5251
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 664
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 662
    })
})

Total tag classes: 27
Examples: ['O', 'B-Area', 'B-Exp', 'B-Features', 'B-HackOrg', 'B-Idus', 'B-OffAct', 'B-Org'] ...


## Step 2 — Tokenizer and subword-label alignment

NER labels come at the **word** level. BERT operates on **subwords**. So "Cobalt Strike" with labels `[B-Tool, I-Tool]` might become `["Co", "##balt", "Strike"]` — three subwords from two words. We need to attach labels to subwords.

Standard recipe (same as CTI 101 notebook 04):

- `tokenizer(..., is_split_into_words=True)` — tells BERT the input is pre-tokenized.
- `.word_ids()` — returns, for each subword, the index of the original word it came from (or `None` for special tokens like `[CLS]`).
- For the **first** subword of each word: assign the word's label.
- For **continuation** subwords: assign `-100` — a magic value PyTorch's loss functions skip.
- For special tokens: also `-100`.

This means each word contributes exactly one prediction to the loss, regardless of how many subwords it spans — cleaner and more stable than alternatives.

In [2]:
from transformers import AutoTokenizer

MODEL_CKPT = 'bert-base-cased'
MAX_LEN = 256   # nb 01 showed p99 sentence length ~150 whitespace tokens; 256 subwords is safe

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)


def tokenize_and_align(batch):
    enc = tokenizer(
        batch['tokens'],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    )
    aligned_labels = []
    for i, word_tags in enumerate(batch['ner_tags']):
        word_ids = enc.word_ids(batch_index=i)
        prev_word = None
        labels = []
        for wid in word_ids:
            if wid is None:
                labels.append(-100)       # [CLS], [SEP], pad
            elif wid != prev_word:
                labels.append(word_tags[wid])   # first subword -> real label
            else:
                labels.append(-100)       # continuation subword -> ignore in loss
            prev_word = wid
        aligned_labels.append(labels)
    enc['labels'] = aligned_labels
    return enc


tokenized = ds.map(tokenize_and_align, batched=True,
                   remove_columns=ds['train'].column_names)
print(tokenized)

Map:   0%|          | 0/5251 [00:00<?, ? examples/s]

Map:   0%|          | 0/664 [00:00<?, ? examples/s]

Map:   0%|          | 0/662 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5251
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 664
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 662
    })
})


### Sanity check the alignment

Print one example side-by-side: subword + its resolved label (or `_IGN_` for -100). If `_IGN_` shows up on continuation pieces and real labels show up on word starts, alignment is working.

In [3]:
ex = tokenized['train'][0]
subwords = tokenizer.convert_ids_to_tokens(ex['input_ids'])

print(f'{"subword":20s}  label')
print('-' * 45)
for sw, lbl in list(zip(subwords, ex['labels']))[:30]:
    tag = '_IGN_' if lbl == -100 else id2label[lbl]
    print(f'{sw:20s}  {tag}')

subword               label
---------------------------------------------
[CLS]                 _IGN_
The                   O
ad                    B-HackOrg
##min                 _IGN_
@                     _IGN_
33                    _IGN_
##8                   _IGN_
has                   O
largely               O
targeted              O
organizations         O
involved              O
in                    O
financial             B-Idus
,                     O
economic              B-Idus
and                   O
trade                 B-Idus
policy                I-Idus
,                     O
typically             O
using                 O
publicly              B-Tool
available             I-Tool
RA                    I-Tool
##T                   _IGN_
##s                   _IGN_
such                  O
as                    O
Po                    B-Tool


## Step 3 — Model and data collator

`DataCollatorForTokenClassification` pads both `input_ids` and `labels` in each batch, using `-100` as the label pad (so padding positions are also ignored by the loss).

In [4]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

collator = DataCollatorForTokenClassification(tokenizer)

print(f'Model: {MODEL_CKPT}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')
print(f'Classifier head: {num_labels} outputs')

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: bert-base-cased
Parameters: 108M
Classifier head: 27 outputs


## Step 4 — Training metrics

For the training loop we compute **token-level** precision / recall / F1 — the simplest honest signal that learning is happening. Token-level metrics are known to be optimistic compared to span-level F1 (which is what notebook 05 reports with `seqeval`), but they're cheap and good enough for watching the validation curve.

We ignore positions where the true label is `-100` (subword continuations and special tokens).

In [5]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    mask = labels != -100
    y_true = labels[mask]
    y_pred = preds[mask]

    # macro averages treat every tag equally — better signal under class imbalance
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

## Step 5 — Training configuration

Defaults chosen for a CPU or single-GPU laptop. On CPU this will still be slow (~30 min for 5 epochs on a few thousand sentences) — if you have a GPU, `Trainer` picks it up automatically.

Knobs worth knowing:
- `learning_rate=2e-5` — the standard BERT fine-tuning rate; rarely needs changing.
- `num_train_epochs=5` — DNRTI is small enough that more epochs risk overfitting. Notebook 05 checks this.
- `per_device_train_batch_size=16` — drop to 8 if you hit OOM.
- `load_best_model_at_end=True` — Trainer keeps the checkpoint with highest validation F1, not the final one.

In [6]:
from transformers import TrainingArguments, Trainer

OUT_DIR = Path('models/ner_dnrti')
OUT_DIR.parent.mkdir(exist_ok=True)

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_195118/1991711009.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Step 6 — Train

Watch two things in the logs:

- **`loss`** should drop steadily and then plateau. If it bounces around, lower the learning rate.
- **`eval_f1`** should climb, then flatten, possibly starting to fall — that's when we start overfitting. `load_best_model_at_end` protects us regardless.

A healthy run on DNRTI typically lands at validation macro-F1 somewhere in the 0.5–0.7 range — much lower than CTI 101's toy task, which is *honest*, not a bug: real CTI data is harder, especially for rare entity types.

In [7]:
train_result = trainer.train()
print('\nFinal training metrics:')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v}')

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.432700,0.355594,0.891149,0.701026,0.581623,0.595826
2,0.281000,0.287766,0.909215,0.716113,0.745391,0.722723
3,0.201400,0.261803,0.918418,0.734372,0.773180,0.745177
4,0.157200,0.265179,0.919725,0.729159,0.816948,0.765394
5,0.146700,0.261662,0.922168,0.739368,0.816937,0.770714



Final training metrics:
  train_runtime: 259.6858
  train_samples_per_second: 101.103
  train_steps_per_second: 6.335
  total_flos: 886998250137654.0
  train_loss: 0.3546990522135355
  epoch: 5.0


## Step 7 — Evaluate on the test set

This is token-level. Notebook 05 will redo evaluation at the **span** level with `seqeval`, which is the number you'd actually report in a paper.

In [8]:
test_metrics = trainer.evaluate(tokenized['test'])
print('Test metrics (token-level):')
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

Test metrics (token-level):
  eval_loss: 0.1934
  eval_accuracy: 0.9430
  eval_precision: 0.8121
  eval_recall: 0.8772
  eval_f1: 0.8415
  eval_runtime: 1.6956
  eval_samples_per_second: 391.6040
  eval_steps_per_second: 12.3850
  epoch: 5.0000


## Step 8 — Save for downstream notebooks

We save the model, the tokenizer, and the tag vocabulary together so notebook 05 and notebook 10 can load everything with one call.

In [9]:
import json

FINAL_DIR = Path('models/ner_dnrti_final')
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

with open(FINAL_DIR / 'tag_vocab.json', 'w') as f:
    json.dump({'tag_list': tag_names, 'tag2id': label2id}, f, indent=2)

print(f'Saved to {FINAL_DIR}:')
for p in sorted(FINAL_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)')

Saved to models/ner_dnrti_final:
  config.json  (0.0 MB)
  model.safetensors  (431.0 MB)
  special_tokens_map.json  (0.0 MB)
  tag_vocab.json  (0.0 MB)
  tokenizer.json  (0.7 MB)
  tokenizer_config.json  (0.0 MB)
  training_args.bin  (0.0 MB)
  vocab.txt  (0.2 MB)


## Step 9 — Quick qualitative check

Numbers tell you whether learning happened; eyeballing predictions tells you whether the model is doing something sensible. Run it on a hand-written sentence and print each word with its predicted tag.

In [10]:
import torch

model.eval()
device = next(model.parameters()).device

sample = "APT29 deployed Cobalt Strike against financial institutions in Europe in 2023."
words = sample.split()

enc = tokenizer(words, is_split_into_words=True, return_tensors='pt', truncation=True, max_length=MAX_LEN)
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = model(**enc).logits[0]
pred_ids = logits.argmax(-1).cpu().tolist()

# Collapse subword predictions back to word-level (take the first subword's label)
word_ids = tokenizer(words, is_split_into_words=True).word_ids()
word_preds = {}
for wid, pid in zip(word_ids, pred_ids):
    if wid is not None and wid not in word_preds:
        word_preds[wid] = pid

print(f'{"word":15s}  predicted tag')
print('-' * 35)
for i, word in enumerate(words):
    tag = id2label[word_preds.get(i, 0)]
    marker = '  <--' if tag != 'O' else ''
    print(f'{word:15s}  {tag}{marker}')

word             predicted tag
-----------------------------------
APT29            B-HackOrg  <--
deployed         O
Cobalt           B-Tool  <--
Strike           I-Tool  <--
against          O
financial        B-Org  <--
institutions     I-Org  <--
in               O
Europe           B-Area  <--
in               O
2023.            B-Time  <--


## Summary

| | What |
|---|---|
| Base model | `bert-base-cased` |
| Task | Token classification (BIO tagging) |
| Dataset | DNRTI (from notebook 01) |
| Tag classes | 27 (13 entities × B/I + O) |
| Saved to | `models/ner_dnrti_final/` |

### What to take away

- The recipe didn't change much from CTI 101 — the **data** did. That's most real-world NER work: the model is a known quantity, the data prep decides the outcome.
- Token-level F1 around 0.5–0.7 on DNRTI is typical. Don't panic; span-level F1 in notebook 05 is the right number to cite.
- Macro F1 will be pulled down by rare entity types (`Purp`, `Features`). Notebook 05 breaks this down per class.

**Next:** notebook 05 runs proper `seqeval`-based span-level evaluation and slices F1 by entity type.